In [2]:
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

INPUT_DIR = Path("/home/wiebe/Desktop/Robotic/AE4317/testing_nn/data/raw")
OUTPUT_DIR = Path("/home/wiebe/Desktop/Robotic/AE4317/testing_nn/data/pre-procest")

EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

TARGET_SIZE = (240, 240)  # (width, height)

def preprocess_image(img: np.ndarray) -> np.ndarray:
    # 1) rotate 90 degrees left
    img = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

    # 2) resize to 240x240
    img = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)

    # 3) denoise
    # fastNlMeansDenoisingColored works on color uint8 images
    img = cv2.fastNlMeansDenoisingColored(img,None,h=10,         # luminance filter strength
        hColor=10,    # color filter strength
        templateWindowSize=7,
        searchWindowSize=21
    )

    # 4) normalize to [0, 1]
    img = img.astype(np.float32) / 255.0

    return img

def save_normalized_image(dst_path: Path, img: np.ndarray):
    """
    Save normalized float image back to viewable PNG/JPG.
    Since image files usually store uint8, we convert back to 0..255 for saving.
    """
    img_to_save = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    cv2.imwrite(str(dst_path), img_to_save)

def process_dataset(input_dir: Path, output_dir: Path):
    files = [p for p in input_dir.rglob("*") if p.suffix.lower() in EXTS]

    if not files:
        print("No image files found.")
        return

    for src_path in tqdm(files, desc="Processing images"):
        rel_path = src_path.relative_to(input_dir)
        dst_path = output_dir / rel_path
        dst_path.parent.mkdir(parents=True, exist_ok=True)

        img = cv2.imread(str(src_path), cv2.IMREAD_COLOR)
        if img is None:
            print(f"Skipping unreadable file: {src_path}")
            continue

        try:
            out = preprocess_image(img)
            save_normalized_image(dst_path, out)
        except Exception as e:
            print(f"Error processing {src_path}: {e}")

if __name__ == "__main__":
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    process_dataset(INPUT_DIR, OUTPUT_DIR)
    print("Done.")

Processing images: 100%|██████████████████████| 625/625 [01:34<00:00,  6.64it/s]

Done.
